# PlantCLEF 2026 — soumission

Notebook de soumission pour Kaggle. La structure est celle de `submission-method.md`. Les cellules 6 et 7 sont celles du protocole. SAM3 est un second modèle : son import, son chemin et son chargement s'ajoutent aux cellules 1, 2 et 4. `predict_species` ne recharge rien.

**Ce run** segmente les fleurs avec SAM3 (prompt `flower`). Pour chaque masque retenu, il forme le carré de côté minimal qui entoure la segmentation, redimensionne ce carré à la taille d'entrée du classifieur, puis calcule le softmax. Le score d'une espèce est le maximum de ses probabilités sur ces carrés. La fonction renvoie les espèces de score au moins `0.1`, par score décroissant. Aucune fleur retenue, ou aucune espèce au-dessus du seuil, donne une liste vide.


In [ ]:
import os, csv
os.environ["KERAS_BACKEND"] = "torch"  # avant keras : SAM3 utilise le backend torch

import numpy as np
import pandas as pd
import torch
import timm
import keras_hub
from PIL import Image
from tqdm import tqdm


In [ ]:
C = "/kaggle/input/competitions/plantclef-2026"
TEST_CSV    = f"{C}/PlantCLEF2025_test.csv"
SPECIES_CSV = f"{C}/species_ids.csv"
IMG_DIR     = f"{C}/PlantCLEF2025_test_images/PlantCLEF2025_test_images"
MODEL_CKPT  = "/kaggle/input/models/juliostat/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3/model_best.pth.tar"
SAM3_PRESET = "/kaggle/input/models/keras/sam3/keras/sam3_pcs/1"

ID_COL  = "quadrat_id"   # vérifié : colonnes du CSV de test = quadrat_id, author, date, license
IMG_EXT = ".jpg"         # vérifié : les 2 105 images sont en .jpg, noms = quadrat_id + extension

def image_path(quadrat_id: str) -> str:
    return f"{IMG_DIR}/{quadrat_id}{IMG_EXT}"

print("GPU :", torch.cuda.is_available())


In [ ]:
df_species = pd.read_csv(SPECIES_CSV)
class_map = df_species["species_id"].to_dict()   # indice de sortie -> species_id

# Contrôle : d'après la page Zenodo des modèles, la sortie 2 correspond à l'espèce 1355870.
assert class_map[2] == 1355870, "species_ids.csv ne suit pas l'ordre des sorties du modèle (class_mapping.txt n'est pas dans les entrées)"


In [ ]:
device = torch.device("cuda")
model = timm.create_model(
    "vit_base_patch14_reg4_dinov2.lvd142m",
    pretrained=False,                 # pas de téléchargement : les poids viennent de MODEL_CKPT
    num_classes=len(df_species),
    checkpoint_path=MODEL_CKPT,
)
model = model.to(device).eval()
data_config = timm.data.resolve_model_data_config(model)

sam3 = keras_hub.models.SAM3PromptableConceptImageSegmenter.from_preset(SAM3_PRESET)


In [ ]:
# Carré minimal de chaque fleur SAM3, redimensionné à l'entrée du classifieur, puis max des softmax.
PROMPT = "flower"
CONF = 0.4        # score SAM3 minimal pour garder une instance
MAX_SIDE = 1200   # côté max envoyé à SAM3 ; le preset réétire ensuite en 1008×1008, sans rognage
SCORE_MIN = 0.1   # probabilité softmax minimale pour retenir une espèce
CROP_BATCH = 16

Image.MAX_IMAGE_PIXELS = None

_RESAMPLE = {
    "nearest": Image.Resampling.NEAREST,
    "bilinear": Image.Resampling.BILINEAR,
    "bicubic": Image.Resampling.BICUBIC,
}


def as_numpy(value):
    if hasattr(value, "detach"):
        value = value.detach().cpu()
    if hasattr(value, "numpy"):
        value = value.numpy()
    return np.asarray(value)


def segment_flowers(image):
    """Masques binaires des fleurs, chacun aligné sur l'image entière."""
    small = image.copy()
    small.thumbnail((MAX_SIDE, MAX_SIDE), Image.Resampling.BILINEAR)
    pixels = np.asarray(small, dtype=np.float32)[None]  # 0–255 : échelle attendue par le preset
    with torch.no_grad():
        out = sam3.predict({"images": pixels, "prompts": [PROMPT]}, verbose=0)

    scores = as_numpy(out["scores"])
    masks = as_numpy(out["masks"])
    if scores.ndim == 2:
        scores = scores[0]
    if masks.ndim == 4:
        masks = masks[0]
    if scores.size and (scores.min() < 0 or scores.max() > 1):
        scores = 1 / (1 + np.exp(-np.clip(scores, -50, 50)))
    if masks.size and (float(masks.min()) < 0 or float(masks.max()) > 1):
        masks = 1 / (1 + np.exp(-np.clip(masks, -50, 50)))
    if scores.size == 0 or not np.any(scores >= CONF):
        return []
    return [mask >= 0.5 for mask in masks[scores >= CONF]]


def squares_from_masks(masks, width, height):
    """Carré minimal entourant chaque masque, en pixels de l'image d'origine (x0, y0, x1, y1)."""
    boxes = []
    seen = set()
    for mask in masks:
        ys, xs = np.nonzero(mask)
        if len(xs) == 0:
            continue
        mask_h, mask_w = mask.shape
        x0 = xs.min() * width / mask_w
        x1 = (xs.max() + 1) * width / mask_w
        y0 = ys.min() * height / mask_h
        y1 = (ys.max() + 1) * height / mask_h
        side = max(x1 - x0, y1 - y0, 1.0)
        left = (x0 + x1) / 2 - side / 2
        top = (y0 + y1) / 2 - side / 2
        if side <= width:
            left = min(max(left, 0.0), width - side)
        if side <= height:
            top = min(max(top, 0.0), height - side)
        box = (float(left), float(top), float(left + side), float(top + side))
        key = tuple(int(round(v)) for v in box)
        if key in seen:
            continue
        seen.add(key)
        boxes.append(box)
    return boxes


def crop_square(img, box):
    """Découpe le carré. Les bords qui dépassent la photo sont répliqués, pour garder un carré."""
    width, height = img.size
    x0, y0, x1, _y1 = box
    side_px = max(1, int(np.ceil(x1 - x0)))
    ix0 = int(np.floor(x0))
    iy0 = int(np.floor(y0))
    arr = np.asarray(img)
    ys = np.clip(np.arange(iy0, iy0 + side_px), 0, height - 1)
    xs = np.clip(np.arange(ix0, ix0 + side_px), 0, width - 1)
    return Image.fromarray(arr[np.ix_(ys, xs)])


def crop_scores(img, boxes):
    """Softmax de chaque carré redimensionné, de forme (n_carrés, n_classes)."""
    _channels, out_h, out_w = data_config["input_size"]
    resample = _RESAMPLE[data_config["interpolation"]]
    mean = np.asarray(data_config["mean"], dtype=np.float32)
    std = np.asarray(data_config["std"], dtype=np.float32)
    parts = []
    for start in range(0, len(boxes), CROP_BATCH):
        crops = []
        for box in boxes[start:start + CROP_BATCH]:
            square = crop_square(img, box).resize((out_w, out_h), resample)
            crop = np.asarray(square, dtype=np.float32) / 255.0
            crops.append((crop - mean) / std)
        batch = np.stack(crops)
        tensor = torch.from_numpy(np.ascontiguousarray(batch)).permute(0, 3, 1, 2).to(device)
        with torch.no_grad(), torch.autocast("cuda"):
            parts.append(torch.softmax(model(tensor).float(), dim=1).cpu())
    return torch.cat(parts).numpy()


def species_from_scores(best) -> list[int]:
    """Espèces de score >= SCORE_MIN, par score décroissant, sans doublon."""
    order = np.argsort(best)[::-1]
    species = []
    seen = set()
    for index in order:
        if best[index] < SCORE_MIN:
            break
        species_id = int(class_map[int(index)])
        if species_id not in seen:
            seen.add(species_id)
            species.append(species_id)
    return species


def predict_species(quadrat_id: str) -> list[int]:
    """Renvoie les espèces retenues pour un quadrat du test. Le modèle est déjà chargé."""
    path = image_path(quadrat_id)
    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    image = Image.open(path).convert("RGB")
    width, height = image.size
    boxes = squares_from_masks(segment_flowers(image), width, height)
    if not boxes:
        return []
    return species_from_scores(crop_scores(image, boxes).max(axis=0))


In [ ]:
def create_submission_file(out_path: str = "/kaggle/working/submission.csv", predict=None) -> pd.DataFrame:
    """Parcourt PlantCLEF2025_test.csv dans l'ordre, appelle predict_species, écrit le CSV."""
    if predict is None:
        predict = predict_species

    ids = pd.read_csv(TEST_CSV, sep=";")[ID_COL].astype(str).tolist()   # ce CSV est séparé par des ;

    rows = []
    for qid in tqdm(ids):
        species = list(dict.fromkeys(int(s) for s in predict(qid)))   # int Python + sans doublons
        rows.append(str(species))

    df_sub = pd.DataFrame({"quadrat_id": ids, "species_ids": rows})
    df_sub.to_csv(out_path, index=False, quoting=csv.QUOTE_ALL)
    return df_sub


In [ ]:
df_sub = create_submission_file()
print(df_sub.shape)
print(df_sub.head())
print("quadrats sans espèce :", (df_sub["species_ids"] == "[]").sum())
